In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 338, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 338 (delta 48), reused 51 (delta 12), pack-reused 224 (from 1)
Receiving objects: 100% (338/338), 5.56 MiB | 3.78 MiB/s, done.
Resolving deltas: 100% (163/163), done.


In [84]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
import json
import os
from sklearn.model_selection import cross_val_score

In [3]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_persona_and_team_data.csv")


# preview
df.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357


In [4]:
df.shape

(480, 218)

In [6]:
# models to train: 12 teams + league-wide
teams = df["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df.copy()
    else:
        df_team = df[df["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics (just MAE + R²)
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# preview
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [7]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
47,team_per_game_PTS,0.056100
35,team_per_game_2P%,0.028499
27,team_per_game_FG,0.026299
137,opp_per_game_2P,0.018609
0,elite_scorer,0.018514
...,...,...
201,opp_shooting_% of FGA by Distance_3P,0.000000
202,opp_shooting_FG% by Distance_2P,0.000000
207,opp_shooting_FG% by Distance_3P,0.000000
204,opp_shooting_FG% by Distance_3-10,0.000000


In [8]:
# drop features from df with team_score importance < 0.001
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.001]["Feature"].tolist()
filtered_df_v1 = df.drop(columns=low_importance_cols, errors="ignore")

In [10]:
# models to train: 12 teams + league-wide
teams = filtered_df_v1["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v1.copy()
    else:
        df_team = filtered_df_v1[filtered_df_v1["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# preview
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [12]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
45,team_per_game_PTS,0.056100
33,team_per_game_2P%,0.028499
25,team_per_game_FG,0.026299
82,opp_per_game_2P,0.018609
0,elite_scorer,0.018514
...,...,...
40,team_per_game_AST,0.003883
31,team_per_game_2P,0.003736
29,team_per_game_3PA,0.003439
90,opp_per_game_PF,0.003209


In [13]:
# drop features from df with team_score importance < 0.005
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.005]["Feature"].tolist()
filtered_df_v2 = df.drop(columns=low_importance_cols, errors="ignore")

In [14]:
# models to train: 12 teams + league-wide
teams = filtered_df_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v2.copy()
    else:
        df_team = filtered_df_v2[filtered_df_v2["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.314095,27.683189,10.343696,8.683189,-1.132368
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400055,25.781090,11.478148,11.791771,-1.399221
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.417870,21.607391,8.215759,6.475475,-0.650818
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.133636,6.471183,6.250748,0.216070
9,PHO,1.229004,16.355804,8.403378,9.037724,0.400211


In [15]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
41,team_per_game_PTS,0.058853
127,opp_per_game_2P,0.036956
25,team_per_game_FG,0.028111
30,team_per_game_2P%,0.026514
115,team_shooting_FG% by Distance_10-16,0.025126
...,...,...
183,opp_shooting_FG%,0.000000
190,opp_shooting_FG% by Distance_2P,0.000000
189,opp_shooting_% of FGA by Distance_3P,0.000000
184,opp_shooting_Dist.,0.000000


In [16]:
# drop features from df with team_score importance < 0.01
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.01]["Feature"].tolist()
filtered_df_v3 = df.drop(columns=low_importance_cols, errors="ignore")

In [17]:
# models to train: 12 teams + league-wide
teams = filtered_df_v3["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v3.copy()
    else:
        df_team = filtered_df_v3[filtered_df_v3["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [18]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.057869
20,team_W,0.050538
15,team_per_game_BLK,0.035029
10,team_per_game_2P%,0.033771
11,team_per_game_FT%,0.032411
39,opp_per_poss_FGA,0.031137
41,opp_shooting_% of FGA by Distance_10-16,0.030505
25,team_shooting_% of FGA by Distance_3-10,0.030012
0,elite_scorer,0.028974
22,team_TOV%.1,0.028882


In [19]:
# drop 'opp_totals_2P' only
filtered_df_v4 = filtered_df_v3.drop(columns=["opp_totals_2P"], errors="ignore")

In [20]:
filtered_df_v4.shape

(480, 48)

In [21]:
# models to train: 12 teams + league-wide
teams = filtered_df_v4["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v4.copy()
    else:
        df_team = filtered_df_v4[filtered_df_v4["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [22]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.057869
20,team_W,0.050538
15,team_per_game_BLK,0.035029
10,team_per_game_2P%,0.033771
11,team_per_game_FT%,0.032411
38,opp_per_poss_FGA,0.031137
40,opp_shooting_% of FGA by Distance_10-16,0.030505
25,team_shooting_% of FGA by Distance_3-10,0.030012
0,elite_scorer,0.028974
22,team_TOV%.1,0.028882


In [26]:
# drop 'opp_totals_2P' only
filtered_df_v5 = filtered_df_v4.drop(columns=["three_point_specialist"], errors="ignore")

In [27]:
# models to train: 12 teams + league-wide
teams = filtered_df_v5["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v5.copy()
    else:
        df_team = filtered_df_v5[filtered_df_v5["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.112991,28.431786,8.927518,5.159397,-0.740086
1,CHI,0.930618,25.069382,9.913876,9.000000,-0.398478
2,CON,1.990623,39.915024,11.439304,6.509377,-1.823733
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.073547,24.424126,8.464477,7.000000,-0.792963
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.335999,16.425278,9.841609,10.624851,0.233447


In [28]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
17,team_per_game_PTS,0.053061
19,team_W,0.044506
39,opp_shooting_% of FGA by Distance_10-16,0.038597
14,team_per_game_BLK,0.035138
37,opp_per_poss_FGA,0.031932
9,team_per_game_2P%,0.031847
40,opp_shooting_FG% by Distance_0-3,0.031678
20,team_ORB%,0.030974
24,team_shooting_% of FGA by Distance_3-10,0.030469
33,opp_per_game_2PA,0.029217


In [29]:
# drop "team_W" from v4
filtered_df_v6 = filtered_df_v4.drop(columns=["team_W"], errors="ignore")

In [30]:
# models to train: 12 teams + league-wide
teams = filtered_df_v6["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v6.copy()
    else:
        df_team = filtered_df_v6[filtered_df_v6["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [31]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.053040
15,team_per_game_BLK,0.036429
10,team_per_game_2P%,0.036074
37,opp_per_poss_FGA,0.036018
39,opp_shooting_% of FGA by Distance_10-16,0.034331
20,team_ORB%,0.033452
24,team_shooting_% of FGA by Distance_3-10,0.032021
0,elite_scorer,0.031700
33,opp_per_game_2PA,0.031511
11,team_per_game_FT%,0.031503


In [32]:
# drop team_W and team_L from df, bring the rest back
filtered_df_v7 = df.drop(columns=["team_W", "team_L"], errors="ignore")

In [33]:
for col in filtered_df_v7:
  print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
team_totals_2P%
team_totals_FT
team_totals

In [40]:
# drop non-feature columns
X = filtered_df_v7.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = filtered_df_v7["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# train baseline model
model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

print(f"Trained model on {X_train.shape[0]} rows and {X_train.shape[1]} features.\n")

Trained model on 360 rows and 210 features.



In [42]:
# get feature names
feature_names = X_train.columns.tolist()

# extract importances
importances_team = model.feature_importances_

# build importance DataFrame
df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
})

# show 0-importance features
zero_importance_cols = df_importances[df_importances["Importance_team_score"] == 0]["Feature"].tolist()
print(f"Found {len(zero_importance_cols)} features with 0 importance:\n{zero_importance_cols}\n")

# drop them to create v7
filtered_df_v7 = filtered_df_v7.drop(columns=zero_importance_cols, errors="ignore")

Found 98 features with 0 importance:
['floor_general', 'stretch_big', 'team_totals_FG', 'team_totals_FGA', 'team_totals_FG%', 'team_totals_3P', 'team_totals_3PA', 'team_totals_3P%', 'team_totals_2P', 'team_totals_2PA', 'team_totals_2P%', 'team_totals_FT', 'team_totals_FTA', 'team_totals_FT%', 'team_totals_DRB', 'team_totals_TRB', 'team_totals_BLK', 'team_totals_TOV', 'team_totals_PF', 'team_totals_PTS', 'team_PL', 'team_MOV', 'team_SOS', 'team_SRS', 'team_NRtg', 'team_FTr', 'team_3PAr', 'team_TS%', 'team_eFG%', 'team_TOV%', 'team_per_poss_FG%', 'team_per_poss_3P', 'team_per_poss_3PA', 'team_per_poss_3P%', 'team_per_poss_2P', 'team_per_poss_2PA', 'team_per_poss_2P%', 'team_per_poss_FT', 'team_per_poss_FTA', 'team_per_poss_FT%', 'team_per_poss_ORB', 'team_per_poss_TRB', 'team_per_poss_AST', 'team_per_poss_STL', 'team_per_poss_BLK', 'team_per_poss_TOV', 'team_per_poss_PTS', 'team_shooting_FG%', 'team_shooting_Dist.', 'team_shooting_% of FGA by Distance_2P', 'team_shooting_% of FGA by Dist

In [44]:
# drop non-feature columns
X = filtered_df_v7.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

all_around_star ⬌ defensive_anchor | correlation = 0.924
team_per_game_3PA ⬌ team_per_game_3P | correlation = 0.965
team_per_game_2PA ⬌ team_per_game_3P | correlation = 0.926
team_per_game_2PA ⬌ team_per_game_3PA | correlation = 0.945
team_per_game_2PA ⬌ team_per_game_2P | correlation = 0.900
team_per_game_FTA ⬌ team_per_game_FT | correlation = 0.915
team_totals_ORB ⬌ team_per_game_ORB | correlation = 1.000
team_totals_AST ⬌ team_per_game_AST | correlation = 1.000
team_totals_STL ⬌ team_per_game_STL | correlation = 1.000
team_PW ⬌ all_around_star | correlation = 0.905
team_ORtg ⬌ team_per_game_PTS | correlation = 0.914
team_ORB% ⬌ team_per_game_ORB | correlation = 0.983
team_ORB% ⬌ team_totals_ORB | correlation = 0.983
team_FT/FGA ⬌ team_per_game_FT | correlation = 0.954
team_FT/FGA ⬌ team_per_game_FTA | correlation = 0.919
team_eFG%.1 ⬌ team_DRtg | correlation = 0.916
team_per_poss_FG ⬌ team_per_game_FG | correlation = 0.932
team_per_poss_DRB ⬌ team_per_game_DRB | correlation = 0.953


In [45]:
# Trim multicollinearity
drop_cols = [
    "team_per_game_3P",
    "team_per_game_3PA",
    "team_per_game_2P",
    "team_per_game_FT",
    "team_totals_ORB",
    "team_totals_AST",
    "team_totals_STL",
    "team_per_game_PTS",
    "team_ORB%",
    "team_per_game_FTA",
    "team_eFG%.1",
    "team_per_game_FG",
    "team_per_game_DRB",
    "team_per_game_PF",
    "opp_per_game_FG",
    "opp_per_game_FG%",  # ← You meant this when referencing team_eFG%.1 links
    "opp_per_game_FT",
    "team_DRB%",
    "team_TOV%.1",
    "opp_totals_2P",
    "opp_totals_TOV",
    "opp_per_poss_FG",
    "opp_per_game_DRB",
    "opp_per_game_TRB",
    "opp_per_game_PF",
    "opp_shooting_% of FGA by Distance_2P"
]

filtered_df_v7 = filtered_df_v7.drop(columns=drop_cols, errors="ignore")


Dropped 26 columns. filtered_df_v8 now has 118 columns.



In [46]:
print(f"Dropped {len(drop_cols)} columns. filtered_df_v7 now has {filtered_df_v7.shape[1]} columns.\n")

Dropped 26 columns. filtered_df_v7 now has 92 columns.



In [47]:
# models to train: 12 teams + league-wide
teams = filtered_df_v7["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v7.copy()
    else:
        df_team = filtered_df_v7[filtered_df_v7["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [48]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
44,team_per_poss_FG,0.076448
29,team_per_game_2P%,0.043564
0,elite_scorer,0.022937
46,team_per_poss_DRB,0.020471
54,team_shooting_FG% by Distance_10-16,0.019813
...,...,...
75,opp_shooting_% of FGA by Distance_0-3,0.005720
79,opp_shooting_FG% by Distance_0-3,0.005678
73,opp_per_poss_TRB,0.005668
7,playmaker,0.005534


In [49]:
# drop team_W and team_L from df, bring the rest back
filtered_df_v8 = df.drop(columns=["team_W", "team_L"], errors="ignore")

In [50]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = filtered_df_v8["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# train baseline model
model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

print(f"Trained model on {X_train.shape[0]} rows and {X_train.shape[1]} features.\n")

Trained model on 360 rows and 210 features.



In [51]:
# get feature names
feature_names = X_train.columns.tolist()

# extract importances
importances_team = model.feature_importances_

# build importance DataFrame
df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
})

# show 0-importance features
zero_importance_cols = df_importances[df_importances["Importance_team_score"] == 0]["Feature"].tolist()
print(f"Found {len(zero_importance_cols)} features with 0 importance:\n{zero_importance_cols}\n")

# drop them to create v7
filtered_df_v8 = filtered_df_v8.drop(columns=zero_importance_cols, errors="ignore")

Found 98 features with 0 importance:
['floor_general', 'stretch_big', 'team_totals_FG', 'team_totals_FGA', 'team_totals_FG%', 'team_totals_3P', 'team_totals_3PA', 'team_totals_3P%', 'team_totals_2P', 'team_totals_2PA', 'team_totals_2P%', 'team_totals_FT', 'team_totals_FTA', 'team_totals_FT%', 'team_totals_DRB', 'team_totals_TRB', 'team_totals_BLK', 'team_totals_TOV', 'team_totals_PF', 'team_totals_PTS', 'team_PL', 'team_MOV', 'team_SOS', 'team_SRS', 'team_NRtg', 'team_FTr', 'team_3PAr', 'team_TS%', 'team_eFG%', 'team_TOV%', 'team_per_poss_FG%', 'team_per_poss_3P', 'team_per_poss_3PA', 'team_per_poss_3P%', 'team_per_poss_2P', 'team_per_poss_2PA', 'team_per_poss_2P%', 'team_per_poss_FT', 'team_per_poss_FTA', 'team_per_poss_FT%', 'team_per_poss_ORB', 'team_per_poss_TRB', 'team_per_poss_AST', 'team_per_poss_STL', 'team_per_poss_BLK', 'team_per_poss_TOV', 'team_per_poss_PTS', 'team_shooting_FG%', 'team_shooting_Dist.', 'team_shooting_% of FGA by Distance_2P', 'team_shooting_% of FGA by Dist

In [53]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.99
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_totals_ORB ⬌ team_per_game_ORB | correlation = 1.000
team_totals_AST ⬌ team_per_game_AST | correlation = 1.000
team_totals_STL ⬌ team_per_game_STL | correlation = 1.000
opp_totals_2P ⬌ opp_per_game_2P | correlation = 1.000
opp_totals_TOV ⬌ opp_per_game_TOV | correlation = 1.000


In [54]:
# columns with perfect 1.000 correlation to other features
drop_cols_v8 = [
    "team_totals_ORB",
    "team_totals_AST",
    "team_totals_STL",
    "opp_totals_2P",
    "opp_totals_TOV"
]

# drop from filtered_df_v8
filtered_df_v8 = filtered_df_v8.drop(columns=drop_cols_v8, errors="ignore")
print(f"Dropped {len(drop_cols_v8)} columns from filtered_df_v8 to update filtered_df_v8.")


Dropped 5 columns from filtered_df_v8 to update filtered_df_v8.


In [55]:
# models to train: 12 teams + league-wide
teams = filtered_df_v8["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v8.copy()
    else:
        df_team = filtered_df_v8[filtered_df_v8["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [56]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
45,team_per_game_PTS,0.058535
33,team_per_game_2P%,0.036860
25,team_per_game_FG,0.028228
42,team_per_game_BLK,0.020093
79,opp_per_game_2P,0.019370
...,...,...
48,team_ORtg,0.000000
74,opp_per_game_FGA,0.000000
73,opp_per_game_FG,0.000000
61,team_shooting_% of FGA by Distance_0-3,0.000000


In [59]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.95
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_3PA ⬌ team_per_game_3P | correlation = 0.965
team_ORB% ⬌ team_per_game_ORB | correlation = 0.983
team_FT/FGA ⬌ team_per_game_FT | correlation = 0.954
team_per_poss_DRB ⬌ team_per_game_DRB | correlation = 0.953
team_per_poss_PF ⬌ team_per_game_PF | correlation = 0.974
opp_per_game_FG% ⬌ team_eFG%.1 | correlation = 0.971
opp_per_game_FT ⬌ team_FT/FGA.1 | correlation = 0.955
opp_per_game_TOV ⬌ team_TOV%.1 | correlation = 0.974
opp_per_poss_FG ⬌ opp_per_game_FG | correlation = 0.974


In [61]:
drop_cols_v9 = [
    "team_per_game_ORB",
    "team_per_game_PF",
    "opp_per_game_TOV",
    "opp_per_game_FG",
    "team_per_game_3P",
    "team_per_game_DRB",
    "opp_per_game_FT",
    "opp_per_game_FG%"
]

filtered_df_v9 = filtered_df_v8.drop(columns=drop_cols_v9, errors="ignore")
print(f"Dropped {len(drop_cols_v9)} columns from filtered_df_v8 to create filtered_df_v9.")

Dropped 8 columns from filtered_df_v8 to create filtered_df_v9.


In [62]:
# models to train: 12 teams + league-wide
teams = filtered_df_v9["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v9.copy()
    else:
        df_team = filtered_df_v9[filtered_df_v9["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [63]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
41,team_per_game_PTS,0.036841
32,team_per_game_2P%,0.033268
25,team_per_game_FG,0.027385
55,team_per_poss_DRB,0.021553
73,opp_per_game_2P,0.021196
...,...,...
28,team_per_game_3PA,0.004981
80,opp_per_game_PF,0.003944
44,team_ORtg,0.003358
30,team_per_game_2P,0.003019


In [64]:
# models to train: 12 teams + league-wide
teams = filtered_df_v7["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v7.copy()
    else:
        df_team = filtered_df_v7[filtered_df_v7["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [65]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
44,team_per_poss_FG,0.076448
29,team_per_game_2P%,0.043564
0,elite_scorer,0.022937
46,team_per_poss_DRB,0.020471
54,team_shooting_FG% by Distance_10-16,0.019813
...,...,...
75,opp_shooting_% of FGA by Distance_0-3,0.005720
79,opp_shooting_FG% by Distance_0-3,0.005678
73,opp_per_poss_TRB,0.005668
7,playmaker,0.005534


In [67]:
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.01]["Feature"].tolist()
print(f"Dropping {len(low_importance_cols)} features with Importance < 0.01 from filtered_df_v7.\n")

Dropping 43 features with Importance < 0.01 from filtered_df_v7.



In [68]:
filtered_df_v10 = filtered_df_v7.drop(columns=low_importance_cols, errors="ignore")
print(f"filtered_df_v10 created with {filtered_df_v10.shape[1]} columns.")

filtered_df_v10 created with 49 columns.


In [69]:
# models to train: 12 teams + league-wide
teams = filtered_df_v10["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v10.copy()
    else:
        df_team = filtered_df_v10[filtered_df_v10["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [70]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
20,team_per_poss_FG,0.096284
38,opp_per_poss_FGA,0.039444
10,team_per_game_2P%,0.034924
0,elite_scorer,0.032104
18,team_ORtg,0.026793
15,team_per_game_BLK,0.026729
22,team_shooting_% of FGA by Distance_3-10,0.026679
11,team_per_game_FT%,0.025068
7,team_per_game_FGA,0.024895
23,team_shooting_% of FGA by Distance_16-3P,0.024879


In [72]:
# drop non-feature columns
X = filtered_df_v10.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.90
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_PW ⬌ all_around_star | correlation = 0.905


In [73]:
# drop team_PW from filtered_df_v10 to create filtered_df_v11
filtered_df_v11 = filtered_df_v10.drop(columns=["team_PW"], errors="ignore")
print("Dropped 'team_PW' from filtered_df_v10 to create filtered_df_v11.")

Dropped 'team_PW' from filtered_df_v10 to create filtered_df_v11.


In [74]:
# models to train: 12 teams + league-wide
teams = filtered_df_v11["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v11.copy()
    else:
        df_team = filtered_df_v11[filtered_df_v11["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [75]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
19,team_per_poss_FG,0.098074
37,opp_per_poss_FGA,0.040177
10,team_per_game_2P%,0.035573
0,elite_scorer,0.032701
17,team_ORtg,0.027291
15,team_per_game_BLK,0.027226
21,team_shooting_% of FGA by Distance_3-10,0.027175
11,team_per_game_FT%,0.025534
7,team_per_game_FGA,0.025358
22,team_shooting_% of FGA by Distance_16-3P,0.025341


In [76]:
# drop non-feature columns
X = filtered_df_v11.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.85
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_2P% ⬌ team_per_game_FG% | correlation = 0.858
team_per_game_BLK ⬌ team_per_game_FT% | correlation = 0.873
team_ORtg ⬌ team_per_game_FG% | correlation = 0.864
team_ORtg ⬌ team_per_game_2P% | correlation = 0.868
team_shooting_FG% by Distance_3-10 ⬌ team_per_game_2P% | correlation = 0.869
opp_per_game_3PA ⬌ opp_per_game_3P | correlation = 0.896
opp_per_poss_FGA ⬌ opp_per_game_FGA | correlation = 0.886


In [77]:
# drop selected columns from filtered_df_v11
drop_cols_v12 = [
    "team_per_game_FG%",
    "team_per_game_2P%",
    "opp_per_game_3P",
    "opp_per_game_FGA"
]

filtered_df_v12 = filtered_df_v11.drop(columns=drop_cols_v12, errors="ignore")
print(f"Dropped {len(drop_cols_v12)} columns from filtered_df_v11 to create filtered_df_v12.")

Dropped 4 columns from filtered_df_v11 to create filtered_df_v12.


In [78]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [79]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
17,team_per_poss_FG,0.077675
33,opp_per_poss_FGA,0.044730
0,elite_scorer,0.039040
13,team_per_game_BLK,0.038586
36,opp_shooting_% of FG Ast'd_3P,0.033695
15,team_ORtg,0.033198
19,team_shooting_% of FGA by Distance_3-10,0.030297
37,opp_shooting_Corner_3P%,0.029107
21,team_shooting_FG% by Distance_3-10,0.028228
7,team_per_game_FGA,0.028040


In [80]:
# drop non-feature columns
X = filtered_df_v12.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.8
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_ORB ⬌ team_per_game_2PA | correlation = 0.832
team_per_game_BLK ⬌ team_per_game_FT% | correlation = 0.873
team_per_poss_FG ⬌ team_ORtg | correlation = 0.810
team_shooting_Corner_%3PA ⬌ team_per_poss_DRB | correlation = 0.848
opp_per_game_2P% ⬌ team_per_poss_DRB | correlation = 0.809
opp_shooting_% of FG Ast'd_3P ⬌ team_per_poss_DRB | correlation = 0.812


In [ ]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

In [81]:
# subset just the league-wide data
df_team = filtered_df_v12.copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# define param grid
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

# initialize base model
base_model = XGBRegressor(
    min_child_weight=1,
    random_state=42,
    verbosity=0
)

# run grid search with 3-fold CV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    n_jobs=-1
)

# fit search
grid_search.fit(X, y)

# best params
print("\nBest parameters found:")
print(grid_search.best_params_)

# best score
print(f"\nBest MAE (CV): {-grid_search.best_score_:.4f}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits

Best parameters found:
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

Best MAE (CV): 8.6212


In [82]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.697418,29.302582,9.042065,5.302582,-0.848202
1,CHI,1.612991,21.142403,10.167221,10.361557,-0.394159
2,CON,1.908020,20.024239,9.117159,6.966129,-0.370147
3,DAL,1.112938,17.930550,7.881152,6.644501,-0.619384
4,IND,0.619118,14.455627,7.547647,8.694450,-0.153583
5,LAS,1.847069,20.152931,9.064523,8.000000,-0.099592
6,LVA,1.232620,18.232620,7.799897,7.000000,-0.380216
7,MIN,0.476959,27.371780,11.558397,10.890656,0.088494
8,NYL,2.971184,17.579269,6.759277,5.275227,0.291995
9,PHO,0.217506,19.994232,11.094264,9.717506,0.048197


In [86]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store results
results = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # model
    model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )

    # MAE (negated)
    neg_mae_scores = cross_val_score(model, X, y, scoring="neg_mean_absolute_error", cv=10)
    mae_scores = -neg_mae_scores

    # R²
    r2_scores = cross_val_score(model, X, y, scoring="r2", cv=10)

    # store both sets
    results.append({
        "Model": team_name,
        "MAE_min": mae_scores.min(),
        "MAE_max": mae_scores.max(),
        "MAE_mean": mae_scores.mean(),
        "MAE_median": np.median(mae_scores),
        "R2_min": r2_scores.min(),
        "R2_max": r2_scores.max(),
        "R2_mean": r2_scores.mean(),
        "R2_median": np.median(r2_scores)
    })

# create summary DataFrame
results_df = pd.DataFrame(results)
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2_min,R2_max,R2_mean,R2_median
0,ATL,3.508930,19.250000,8.104744,6.566471,-3.228904,-0.000144,-0.817832,-0.337204
1,CHI,2.061405,13.750000,7.922522,7.160944,-9.633927,-0.000001,-1.352348,-0.196554
2,CON,4.178896,13.000000,8.362984,8.262123,-4.199431,0.136269,-0.684993,-0.399390
3,DAL,1.250000,14.546692,9.082028,9.429706,-2.911178,0.014238,-0.755091,-0.214249
4,IND,3.073570,13.000000,8.157331,8.432936,-2.711883,0.201647,-0.962626,-0.628400
5,LAS,4.640919,17.774118,8.609738,8.054203,-1.410318,-0.001252,-0.462690,-0.278375
6,LVA,2.301861,12.706177,7.498015,7.078327,-3.020658,-0.001343,-0.988823,-0.851452
7,MIN,5.006443,11.943167,8.537713,8.934809,-2.460590,0.100193,-0.634334,-0.321766
8,NYL,5.648062,11.342678,7.581281,7.009244,-24.847496,0.396241,-2.576591,0.062027
9,PHO,5.601460,12.083982,8.875272,9.510556,-4.268547,0.268144,-0.485871,0.004899
